# Probabilistic SMS Spam Detection
**BAMAT207 Probability and Statistics**  
**Student:** Kushal Bagla  
**Roll number:** 25BAI0044  
**Topic:** Probability and Random Variables  
**Submission deadline:** 27 October 2026

This project uses probability and a Multinomial Naive Bayes classifier to distinguish spam from genuine SMS messages. This stage completes **Activity 1 Dataset Handling** and records the cleaned dataset summary. Probability analysis, charts and model evaluation will follow in later stages.

## Running this notebook
Run cells in order from the top. Locally, open this notebook from the repository root using the Python environment described in README.md. The source dataset is included in `data/raw`. On Google Colab, upload this notebook and run all cells; it downloads the same official UCI data if the local file is absent. Pandas is required and is normally preinstalled in Colab. The loader and cleaning functions are included here so the notebook can run independently of the repository's Python modules.

Outputs are saved relative to the notebook's current working directory. No model results are claimed at this stage.

## Dataset source
Almeida, T. and Hidalgo, J. (2011). *SMS Spam Collection*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84

Source: https://archive.ics.uci.edu/dataset/228/sms+spam+collection  
Licence listed by UCI: [Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/).

The original file contains one SMS record per physical line, with a label followed by a tab and the message. We split only at the first tab and preserve quotation marks and any further tabs as message content. A SHA-256 check ensures that the file matches the verified source download.

## 1 Dataset Handling
### Load the SMS messages and labels
The following reusable functions match `src/data.py`. Cleaning removes records with missing or blank labels/messages, checks for unexpected or conflicting labels, removes exact duplicate message text and encodes Ham = 0 and Spam = 1. Case, punctuation and whitespace in nonblank SMS text are preserved. Deduplication happens before the future training/test split to avoid placing identical messages in both sets.

In [1]:
"""Load the official SMS Spam Collection without CSV quoting assumptions."""
from hashlib import sha256
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd

DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
SOURCE_SHA256 = "7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d"


def download_dataset(path: Path) -> Path:
    """Download the official file if absent; verify cached or downloaded bytes."""
    path = Path(path)
    if path.exists():
        content = path.read_bytes()
    else:
        with urlopen(DATA_URL, timeout=60) as response:
            with ZipFile(BytesIO(response.read())) as archive:
                content = archive.read("SMSSpamCollection")
    if sha256(content).hexdigest() != SOURCE_SHA256:
        raise ValueError("Dataset checksum differs from the verified UCI source.")
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    return path


def load_dataset(path: Path) -> pd.DataFrame:
    """Split each physical record at its first tab, preserving SMS text."""
    records = []
    for line_number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if "\t" not in line:
            raise ValueError(f"Source line {line_number} has no label separator.")
        label, message = line.split("\t", 1)
        records.append({"source_line": line_number, "label": label, "message": message})
    return pd.DataFrame(records, columns=["source_line", "label", "message"])


def clean_dataset(raw: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Remove missing/blank records and exact duplicate SMS messages.

    Preserve case, punctuation and whitespace in nonblank SMS text. Detect
    conflicting labels before deduplication rather than choosing one silently.
    """
    required = {"source_line", "label", "message"}
    if not required.issubset(raw.columns):
        raise ValueError(f"Required columns: {sorted(required)}")
    missing = raw[["label", "message"]].isna()
    blank = raw[["label", "message"]].apply(lambda col: col.fillna("").str.strip().eq(""))
    invalid = (missing | blank).any(axis=1)
    valid = raw.loc[~invalid].copy()
    unknown = set(valid["label"]) - {"spam", "ham"}
    if unknown:
        raise ValueError(f"Unexpected labels: {sorted(unknown)}")
    conflicts = valid.groupby("message")["label"].nunique().gt(1)
    if conflicts.any():
        raise ValueError("Identical messages have conflicting labels; inspect the source.")
    duplicate_count = int(valid.duplicated(subset="message").sum())
    clean = valid.drop_duplicates(subset="message", keep="first").copy()
    clean["target"] = clean["label"].map({"ham": 0, "spam": 1}).astype("int64")
    clean = clean.reset_index(drop=True)
    summary = {
        "raw_records": int(len(raw)),
        "missing_values": {key: int(value) for key, value in missing.sum().items()},
        "blank_non_null_values": {
            key: int(value) for key, value in (blank & ~missing).sum().items()
        },
        "removed_missing_or_blank_records": int(invalid.sum()),
        "removed_duplicate_messages": duplicate_count,
        "clean_records": int(len(clean)),
        "raw_class_counts": {key: int(value) for key, value in raw["label"].value_counts().items()},
        "clean_class_counts": {key: int(value) for key, value in clean["label"].value_counts().items()},
        "label_encoding": {"ham": 0, "spam": 1},
        "duplicate_rule": "Exact message text; retain the first occurrence",
    }
    return clean, summary


In [2]:
raw_path = download_dataset(Path("data/raw/SMSSpamCollection"))
raw = load_dataset(raw_path)
print(f"Loaded {len(raw):,} records from the verified UCI file.")
print("Raw class counts:")
print(raw["label"].value_counts().to_string())

Loaded 5,574 records from the verified UCI file.
Raw class counts:
label
ham     4827
spam     747


### Check missing values and remove duplicate messages
Missing values and whitespace-only fields are counted separately. Exact duplicates retain their first occurrence. The source line number is retained so each cleaned record can be traced back to the raw file.

In [3]:
import json

clean, summary = clean_dataset(raw)
print(json.dumps(summary, indent=2))
assert clean["message"].is_unique
assert not clean[["label", "message", "target"]].isna().any().any()
assert set(clean["target"].unique()) == {0, 1}
assert len(raw) == len(clean) + summary["removed_missing_or_blank_records"] + summary["removed_duplicate_messages"]
print("Integrity checks passed.")

{
  "raw_records": 5574,
  "missing_values": {
    "label": 0,
    "message": 0
  },
  "blank_non_null_values": {
    "label": 0,
    "message": 0
  },
  "removed_missing_or_blank_records": 0,
  "removed_duplicate_messages": 403,
  "clean_records": 5171,
  "raw_class_counts": {
    "ham": 4827,
    "spam": 747
  },
  "clean_class_counts": {
    "ham": 4518,
    "spam": 653
  },
  "label_encoding": {
    "ham": 0,
    "spam": 1
  },
  "duplicate_rule": "Exact message text; retain the first occurrence"
}
Integrity checks passed.


### Cleaned dataset summary
The original dataset contains **5,574 records**: 4,827 ham and 747 spam. There are no missing or blank labels/messages. Removing **403 duplicate messages** leaves **5,171 unique messages**, comprising **4,518 ham** and **653 spam**. These cleaned records will be the basis of the probability calculations and later model split.

Deduplication uses exact message text. Similar messages with different wording or formatting remain distinct; this rule is explicit and reproducible. No labels conflict for identical messages in this source.

In [4]:
counts = pd.DataFrame({
    "Original": raw["label"].value_counts(),
    "Cleaned": clean["label"].value_counts(),
}).reindex(["ham", "spam"])
counts["Duplicates removed"] = counts["Original"] - counts["Cleaned"]
print(counts.to_string())
print("\nNumeric labels: ham = 0; spam = 1")

       Original  Cleaned  Duplicates removed
label                                       
ham        4827     4518                 309
spam        747      653                  94

Numeric labels: ham = 0; spam = 1


### Save the cleaned dataset and summary
The CSV includes `source_line`, `label`, `message` and `target`. The JSON summary records all cleaning counts and the duplicate rule. CSV quoting preserves messages containing commas, quotes and tabs; load the exported file with `pandas.read_csv`.

In [5]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("reports").mkdir(parents=True, exist_ok=True)
clean.to_csv("data/processed/sms_cleaned.csv", index=False)
Path("reports/dataset_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
restored = pd.read_csv("data/processed/sms_cleaned.csv", keep_default_na=False)
pd.testing.assert_frame_equal(clean, restored)
print("Saved cleaned dataset and summary; CSV round-trip check passed.")

Saved cleaned dataset and summary; CSV round-trip check passed.
